# 05 — DifIISR Target Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shelly-serafimovich/target-aware-infrared-sr/blob/main/notebooks/05_difiisr_target_dynamics.ipynb)

Diagnostic only: **no guidance and no model modification**. We inspect target-local SCR and calibrated Hoyer in the frozen Original DifIISR one-step output.

We begin with a small deterministic sample: up to 15 validation targets from each LR size bin `≤2`, `2–4`, `4–8`. The frozen LR set, pinned DifIISR commit, checkpoint, and seed are the same as the baseline.


## 1. Setup

Run on a GPU runtime. This reuses the isolated DifIISR environment from the baseline design and does **not** regenerate LR.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,subprocess,json,textwrap
PROJECT_REPO='https://github.com/shelly-serafimovich/target-aware-infrared-sr.git'
DIFIISR_REPO='https://github.com/zirui0625/DifIISR.git'
DIFIISR_COMMIT='09ca97ea48d481656dd8090e84099c963059ac41'
SEED=12345; N_PER_BIN=15
DATASET_ROOT=Path('/content/drive/MyDrive/thermal-drone-dataset')
LR_DIR=DATASET_ROOT/'difiisr_baseline'/'valid'/'LR_difiisr_train_degradation_v2'
OUT=DATASET_ROOT/'difiisr_target_dynamics'; OUT.mkdir(parents=True,exist_ok=True)
project_dir=Path('/content/target-aware-infrared-sr'); difiisr_dir=Path('/content/DifIISR')
def run(cmd,cwd=None): print('$',' '.join(map(str,cmd))); subprocess.run(list(map(str,cmd)),cwd=cwd,check=True)
if not project_dir.exists(): run(['git','clone',PROJECT_REPO,project_dir])
if not difiisr_dir.exists(): run(['git','clone',DIFIISR_REPO,difiisr_dir])
run(['git','fetch','--all'],cwd=difiisr_dir); run(['git','checkout','--detach',DIFIISR_COMMIT],cwd=difiisr_dir)
ENV=Path('/content/difiisr-env'); PY=str(ENV/'bin'/'python')
if not Path(PY).exists():
    run([sys.executable,'-m','pip','install','-q','uv']); run([sys.executable,'-m','uv','venv','--python','3.10',ENV])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'torch==2.1.1','torchvision==0.16.1','xformers==0.0.23','numpy==1.25.2','scipy==1.9.3','packaging==24.2','setuptools==69.5.1'])
    run([sys.executable,'-m','uv','pip','install','--python',PY,'pyiqa==0.1.12','basicsr==1.4.2','opencv-python','matplotlib','timm','pandas','pillow','scikit-learn','scikit-image','lpips','loguru','omegaconf','six','tqdm','albumentations','einops','imageio','gdown'])
weights=difiisr_dir/'weights'; weights.mkdir(exist_ok=True); ckpt=weights/'DifIISR.pth'
if not ckpt.exists(): run([PY,'-m','gdown','1PhRvk1Dlp3CCrPkrxRfNbNZ3fNgviDVZ','-O',ckpt])
print('Frozen LR:',LR_DIR); print('Output:',OUT)


## 2. Select tiny targets

Selection uses the corrected validation YOLO labels and actual LR dimensions. The same image can contribute more than one target if applicable.


In [ ]:
import pandas as pd, numpy as np
ANN=project_dir/'annotations'/'corrected_yolo'/'valid'/'labels'
from PIL import Image
rows=[]
for lp in sorted(ANN.glob('*.txt')):
    ip=next(iter(LR_DIR.glob(lp.stem+'.*')),None)
    if ip is None: continue
    with Image.open(ip) as im: W,H=im.size
    for idx,line in enumerate([x for x in lp.read_text().splitlines() if x.strip()]):
        p=line.split();
        if len(p)<5: continue
        _,xc,yc,bw,bh=map(float,p[:5]); md=max(bw*W,bh*H)
        b='≤2 px' if md<=2 else '2–4 px' if md<=4 else '4–8 px' if md<=8 else None
        if b: rows.append(dict(image=ip.name,target_idx=idx,xc=xc,yc=yc,bw=bw,bh=bh,max_dim_lr=md,lr_size_bin=b))
targets=pd.DataFrame(rows)
sample=(targets.groupby('lr_size_bin',group_keys=False).apply(lambda g:g.sample(min(N_PER_BIN,len(g)),random_state=SEED)).reset_index(drop=True))
display(sample.groupby('lr_size_bin').size().to_frame('selected'))
manifest=OUT/'pilot_targets.csv'; sample.to_csv(manifest,index=False); print(manifest)


## 3. Instrument the actual Original DifIISR inference

The official pinned `inference.py` calls `sampler.inference(..., one_step=True)`, so the frozen baseline stops after the highest diffusion timestep and uses `pred_xstart` as the final latent sample.

The official sampler also reflect-pads each LR image to a multiple of 64 before encoding. This is required by the Swin blocks and is why we must reproduce that padding here. After decoding, we crop back to the original ×4 output size exactly like `sample_func`.


In [ ]:
script=OUT/'run_target_dynamics.py'
script.write_text(r'''import argparse,sys,random,math
from pathlib import Path
import numpy as np,pandas as pd,torch,torch.nn.functional as F
from PIL import Image
from omegaconf import OmegaConf
sys.path.insert(0,'/content/DifIISR')
from sampler import DifIISRSampler

ap=argparse.ArgumentParser(); ap.add_argument('--manifest'); ap.add_argument('--lr'); ap.add_argument('--out'); ap.add_argument('--config'); ap.add_argument('--seed',type=int,default=12345); a=ap.parse_args()
random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed); torch.cuda.manual_seed_all(a.seed)
cfg=OmegaConf.load(a.config); cfg.model.ckpt_path='/content/DifIISR/weights/DifIISR.pth'; cfg.diffusion.params.sf=4; cfg.autoencoder.ckpt_path='/content/DifIISR/weights/autoencoder_vq_f4.pth'
if not Path(cfg.autoencoder.ckpt_path).exists():
 from basicsr.utils.download_util import load_file_from_url
 load_file_from_url(url='https://github.com/zsyOAOA/ResShift/releases/download/v2.0/autoencoder_vq_f4.pth',model_dir='/content/DifIISR/weights',progress=True,file_name='autoencoder_vq_f4.pth')
s=DifIISRSampler(cfg,chop_size=512,chop_stride=448,chop_bs=1,use_fp16=True,seed=a.seed,ddim=True)
df=pd.read_csv(a.manifest); lrdir=Path(a.lr); out=Path(a.out); out.mkdir(parents=True,exist_ok=True)

def metric(im,xc,yc,outer,core,noise_floor=1e-6):
 H,W=im.shape; cx=int(round(xc*W)); cy=int(round(yc*H)); ho=outer//2; hc=core//2; x0,x1=cx-ho,cx+ho+1; y0,y1=cy-ho,cy+ho+1
 if x0<0 or y0<0 or x1>W or y1>H:return None
 p=im[y0:y1,x0:x1].astype(np.float32); c0=ho-hc; c1=c0+core; c=p[c0:c1,c0:c1]; m=np.ones_like(p,dtype=bool); m[c0:c1,c0:c1]=False; ring=p[m]
 rs=float(ring.std()); scr=(float(c.max())-float(ring.mean()))/max(rs,noise_floor)
 x=np.maximum(0.,c.ravel()-float(ring.mean())); l1=float(np.abs(x).sum()); l2=float(np.linalg.norm(x)); n=x.size
 if l2<=0:h=0.
 else:
  raw=(np.sqrt(n)-l1/l2)/(np.sqrt(n)-1.); expected=np.sqrt(n/2.)*rs; excess=max(0.,l2-expected); h=float(raw*excess/(excess+expected+1e-6))
 return scr,h

rows=[]
for k,r in df.iterrows():
 ip=lrdir/r.image; arr=np.asarray(Image.open(ip).convert('RGB'),dtype=np.float32)/255.; y=torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).cuda()
 y=(y-.5)/.5
 ori_h,ori_w=y.shape[-2:]
 if ori_h>512 or ori_w>512: print('SKIP_CHOP',r.image); continue
 # Reproduce DifIISR.sample_func preprocessing: reflect-pad LR to multiples of 64.
 desired=64; pad_h=(math.ceil(ori_h/desired))*desired-ori_h; pad_w=(math.ceil(ori_w/desired))*desired-ori_w
 if pad_h or pad_w: y=F.pad(y,(0,pad_w,0,pad_h),mode='reflect')
 model_kwargs={'lq':y} if s.configs.model.params.cond_lq else None
 z_y=s.base_diffusion.encode_first_stage(y,s.autoencoder,up_sample=True); noise=torch.randn_like(z_y); z=s.base_diffusion.prior_sample(z_y,noise)
 i=s.base_diffusion.num_timesteps-1; t=torch.tensor([i],device=y.device)
 with torch.no_grad():
  o=s.base_diffusion.ddim_sample(model=s.model,x=z,y=z_y,t=t,clip_denoised=False,model_kwargs=model_kwargs)
  pred=s.base_diffusion.decode_first_stage(o['pred_xstart'],s.autoencoder).clamp(-1,1)
 pred=pred[:,:,:ori_h*4,:ori_w*4]
 gray=((pred[0].float().mean(0).cpu().numpy()+1.)*.5*255.).clip(0,255)
 mh=metric(gray,r.xc,r.yc,61,31)
 if mh is not None: rows.append(dict(image=r.image,target_idx=int(r.target_idx),lr_size_bin=r.lr_size_bin,max_dim_lr=r.max_dim_lr,step=0,timestep=i,scr=mh[0],hoyer=mh[1]))
 print(k+1,'/',len(df),r.image)
pd.DataFrame(rows).to_csv(out/'pilot_target_dynamics.csv',index=False)
print('DONE',len(rows))
''')
cmd=[PY,'-u',str(script),'--manifest',str(manifest),'--lr',str(LR_DIR),'--out',str(OUT),'--config',str(difiisr_dir/'configs'/'DifIISR_test.yaml'),'--seed',str(SEED)]
print('Ready to run:', ' '.join(map(str,cmd)))


## 4. Run the pilot


In [ ]:
env=os.environ.copy(); env['MPLBACKEND']='Agg'; env['CUDA_VISIBLE_DEVICES']='0'
result=subprocess.run(list(map(str,cmd)),cwd=str(difiisr_dir),env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout, flush=True)
if result.returncode!=0: raise RuntimeError(f'Target-dynamics pilot failed with exit code {result.returncode}. The real subprocess error is printed directly above.')
csv=OUT/'pilot_target_dynamics.csv'; assert csv.exists(); dynamics=pd.read_csv(csv); print('Rows:',len(dynamics)); display(dynamics.head())


## 5. Compare frozen LR with Original DifIISR SR

The official baseline is one-step, so we compare the same pilot targets before and after Original DifIISR. LR uses 15×15 outer / 7×7 core; SR uses 61×61 outer / 31×31 core. SCR is calibrated separately at each scale from the 5th percentile of positive ring standard deviations.


In [ ]:
from PIL import Image
import numpy as np, pandas as pd
SR_DIR=DATASET_ROOT/'difiisr_baseline'/'valid'/'SR_difiisr_train_degradation_v2'
assert SR_DIR.exists(),SR_DIR
def patch(im,xc,yc,outer,core):
 H,W=im.shape; cx=int(round(xc*W)); cy=int(round(yc*H)); ho=outer//2; hc=core//2
 x0,x1=cx-ho,cx+ho+1; y0,y1=cy-ho,cy+ho+1
 if x0<0 or y0<0 or x1>W or y1>H:return None
 p=im[y0:y1,x0:x1].astype(np.float32); c0=ho-hc; c1=c0+core; c=p[c0:c1,c0:c1]
 m=np.ones_like(p,dtype=bool); m[c0:c1,c0:c1]=False
 return c,p[m]
def metrics(c,ring,floor):
 mu=float(ring.mean()); sd=float(ring.std()); scr=(float(c.max())-mu)/max(sd,floor)
 x=np.maximum(0.,c.ravel()-mu); l1=float(abs(x).sum()); l2=float(np.linalg.norm(x)); n=x.size
 if l2<=0:h=0.
 else:
  raw=(np.sqrt(n)-l1/l2)/(np.sqrt(n)-1.); exp=np.sqrt(n/2.)*sd; exc=max(0.,l2-exp); h=float(raw*exc/(exc+exp+1e-6))
 return scr,h
pairs=[]
for _,r in sample.iterrows():
 lp=LR_DIR/r.image; sp=SR_DIR/r.image
 if not lp.exists() or not sp.exists():continue
 lr=np.asarray(Image.open(lp).convert('L'),dtype=np.float32); sr=np.asarray(Image.open(sp).convert('L'),dtype=np.float32)
 a=patch(lr,r.xc,r.yc,15,7); b=patch(sr,r.xc,r.yc,61,31)
 if a is not None and b is not None:pairs.append((r,a,b))
lr_sd=np.array([a[1].std() for _,a,_ in pairs]); sr_sd=np.array([b[1].std() for _,_,b in pairs])
LR_F=float(np.percentile(lr_sd[lr_sd>0],5)); SR_F=float(np.percentile(sr_sd[sr_sd>0],5))
print('Paired:',len(pairs),'| LR floor:',LR_F,'| SR floor:',SR_F)
rows=[]
for r,a,b in pairs:
 ls,lh=metrics(*a,LR_F); ss,sh=metrics(*b,SR_F)
 rows.append(dict(image=r.image,target_idx=int(r.target_idx),lr_size_bin=r.lr_size_bin,max_dim_lr=r.max_dim_lr,lr_scr=ls,sr_scr=ss,delta_scr=ss-ls,lr_hoyer=lh,sr_hoyer=sh,delta_hoyer=sh-lh))
compare=pd.DataFrame(rows); compare.to_csv(OUT/'pilot_lr_vs_original_sr.csv',index=False); display(compare.head())


## 6. Size-stratified paired summary

This is a small diagnostic pilot, not the final statistical result.


In [ ]:
order=['≤2 px','2–4 px','4–8 px']; out=[]
for b in order:
 g=compare[compare.lr_size_bin==b]
 if len(g): out.append({'size_bin':b,'n':len(g),'LR SCR median':g.lr_scr.median(),'SR SCR median':g.sr_scr.median(),'ΔSCR median':g.delta_scr.median(),'SCR increased %':100*(g.delta_scr>0).mean(),'LR Hoyer median':g.lr_hoyer.median(),'SR Hoyer median':g.sr_hoyer.median(),'ΔHoyer median':g.delta_hoyer.median(),'Hoyer increased %':100*(g.delta_hoyer>0).mean()})
summary=pd.DataFrame(out); display(summary.round(4)); summary.to_csv(OUT/'pilot_lr_vs_original_sr_summary.csv',index=False)


In [ ]:
import matplotlib.pyplot as plt
for m,label in [('scr','SCR'),('hoyer','Calibrated Hoyer')]:
 xs=[]; a=[]; b=[]
 for j,bin_ in enumerate(order):
  g=compare[compare.lr_size_bin==bin_]
  if len(g): xs.append(j); a.append(g['lr_'+m].median()); b.append(g['sr_'+m].median())
 fig,ax=plt.subplots(figsize=(7,4)); ax.plot(xs,a,marker='o',label='Frozen LR'); ax.plot(xs,b,marker='o',label='Original DifIISR SR')
 ax.set_xticks(xs,[order[j] for j in xs]); ax.set_ylabel(label); ax.set_xlabel('Target size in frozen LR'); ax.set_title(label+': LR vs Original DifIISR SR'); ax.legend(); ax.grid(alpha=.25); plt.show()


## 7. Same-image background control

For every paired target, sample one valid background location in the same image, avoiding corrected GT boxes. Then compare LR→SR metric changes at targets versus background.


In [ ]:
import random
rng=random.Random(SEED); gt={}
for lp in ANN.glob('*.txt'):
 boxes=[]
 for line in lp.read_text().splitlines():
  p=line.split()
  if len(p)>=5: boxes.append(tuple(map(float,p[1:5])))
 gt[lp.stem]=boxes
def overlap(cx,cy,W,H,boxes):
 half=9; x0,x1=cx-half,cx+half+1; y0,y1=cy-half,cy+half+1
 for xc,yc,bw,bh in boxes:
  gx0,gx1=(xc-bw/2)*W,(xc+bw/2)*W; gy0,gy1=(yc-bh/2)*H,(yc+bh/2)*H
  if x0<gx1 and x1>gx0 and y0<gy1 and y1>gy0:return True
 return False
bg=[]
for _,r in compare.iterrows():
 lr=np.asarray(Image.open(LR_DIR/r.image).convert('L'),dtype=np.float32); sr=np.asarray(Image.open(SR_DIR/r.image).convert('L'),dtype=np.float32); H,W=lr.shape
 found=None
 for _ in range(200):
  cx=rng.randint(7,W-8); cy=rng.randint(7,H-8)
  if overlap(cx,cy,W,H,gt.get(Path(r.image).stem,[])):continue
  a=patch(lr,cx/W,cy/H,15,7); b=patch(sr,cx/W,cy/H,61,31)
  if a is not None and b is not None:found=(a,b);break
 if found:
  a,b=found; ls,lh=metrics(*a,LR_F); ss,sh=metrics(*b,SR_F)
  bg.append(dict(image=r.image,lr_size_bin=r.lr_size_bin,delta_scr=ss-ls,delta_hoyer=sh-lh))
background=pd.DataFrame(bg); background.to_csv(OUT/'pilot_same_image_background.csv',index=False)
print('Matched background:',len(background)); display(background.head())


## 8. Target change versus background change

A positive target change alone is not enough: the background control helps determine whether DifIISR is changing the target cue more than the surrounding image generally.


In [ ]:
rows=[]
for b in order:
 t=compare[compare.lr_size_bin==b]; q=background[background.lr_size_bin==b]
 if len(t) and len(q): rows.append({'size_bin':b,'target_n':len(t),'background_n':len(q),'target ΔSCR median':t.delta_scr.median(),'background ΔSCR median':q.delta_scr.median(),'target ΔHoyer median':t.delta_hoyer.median(),'background ΔHoyer median':q.delta_hoyer.median()})
control=pd.DataFrame(rows); display(control.round(4)); control.to_csv(OUT/'pilot_target_vs_background_change.csv',index=False)
print('Saved to:',OUT)
